# Gộp các batch Kaggle thành success.jsonl và errors.jsonl

Chạy lần lượt 3 cell trên Google Colab. Sửa `BATCH_FOLDER` nếu cần; folder được chia sẻ cần có shortcut trong My Drive.

Notebook đọc tất cả `worker_*_batch_*.jsonl` đang có trong folder gốc (mọi worker), rồi phân loại theo `success`: `true` vào `success.jsonl`, `false` vào `errors.jsonl`. Hai file nằm trong `results/DD-MM-YYYY`, ngày chạy theo giờ Việt Nam. Không lọc batch theo ngày tạo và không xóa batch gốc, để Kaggle vẫn resume được.

Chạy lại sẽ tạo lại hai file tổng hợp từ các batch hiện có, không append nên không nhân đôi dữ liệu vì chạy lại. Các bản ghi trùng `doc_id` vốn có trong các batch vẫn được giữ, bao gồm lịch sử retry. Notebook đọc từng dòng để tránh nạp toàn bộ dữ liệu vào RAM. Dòng lỗi JSON hoặc thiếu trường boolean `success` sẽ dừng việc gộp thay vì bỏ dữ liệu.

Chạy lại cell cấu hình nếu đã qua ngày mới. Kết quả chỉ bao gồm các batch đã có khi cell gộp bắt đầu.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

# Folder chứa các file worker_XX_batch_*.jsonl do Kaggle tạo.
BATCH_FOLDER = Path("/content/drive/MyDrive/lucraftmeds_crawl")

TODAY = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%d-%m-%Y")
OUTPUT_FOLDER = BATCH_FOLDER / "results" / TODAY

if not BATCH_FOLDER.is_dir():
    raise FileNotFoundError(f"Không tìm thấy folder: {BATCH_FOLDER}")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print("Folder đích:", OUTPUT_FOLDER)

In [ ]:
import json

batch_files = sorted(
    path for path in BATCH_FOLDER.glob("worker_*_batch_*.jsonl") if path.is_file()
)
if not batch_files:
    raise FileNotFoundError("Không tìm thấy file worker_*_batch_*.jsonl trong BATCH_FOLDER.")

success_path = OUTPUT_FOLDER / "success.jsonl"
errors_path = OUTPUT_FOLDER / "errors.jsonl"
success_temp = OUTPUT_FOLDER / "success.jsonl.part"
errors_temp = OUTPUT_FOLDER / "errors.jsonl.part"
counts = {True: 0, False: 0}

try:
    with success_temp.open("w", encoding="utf-8") as success_file, \
         errors_temp.open("w", encoding="utf-8") as errors_file:
        for batch_path in batch_files:
            with batch_path.open("r", encoding="utf-8") as batch_file:
                for line_number, line in enumerate(batch_file, 1):
                    if not line.strip():
                        continue
                    try:
                        record = json.loads(line)
                    except ValueError as exc:
                        raise ValueError(f"JSON lỗi: {batch_path.name}, dòng {line_number}") from exc
                    if not isinstance(record, dict) or not isinstance(record.get("success"), bool):
                        raise ValueError(f"Thiếu success true/false: {batch_path.name}, dòng {line_number}")
                    ok = record["success"]
                    output = success_file if ok else errors_file
                    output.write(line.rstrip("\r\n") + "\n")
                    counts[ok] += 1

    # Chỉ thay file kết quả sau khi đọc và phân loại xong tất cả batch.
    success_temp.replace(success_path)
    errors_temp.replace(errors_path)
finally:
    success_temp.unlink(missing_ok=True)
    errors_temp.unlink(missing_ok=True)

print(f"Đã gộp {len(batch_files)} batch.")
print(f"Success: {counts[True]:,} dòng → {success_path}")
print(f"Errors:  {counts[False]:,} dòng → {errors_path}")
